# Lesson 1: JOINs

**Week 2 · Data Engineering Course**

---

**Prerequisites:** Lab 2 complete — retail database loaded with all 5 tables.

**What you will learn:**
- Why data is split across multiple tables
- INNER JOIN — only matching rows
- LEFT JOIN — all rows from the left table
- Joining more than two tables
- Common JOIN mistakes and how to avoid them

**All SQL runs in pgAdmin → Tools → Query Tool → select `de_course`**

---

## 2.1 Why JOINs Exist

In Week 1, every query touched one table: `employees`. Real databases split data across many tables.

In the retail database:
- `customers` stores who the customer is
- `orders` stores when they bought something
- `order_items` stores what they bought and how many
- `products` stores what each product is
- `salespeople` stores who handled the sale

If you want to answer *'which customer bought which product?'*, the answer lives across four tables. A JOIN stitches them together.

**The key idea:** tables are linked by shared columns — usually a primary key in one table that appears as a foreign key in another.

```
customers.customer_id  ←→  orders.customer_id
orders.order_id        ←→  order_items.order_id
products.product_id    ←→  order_items.product_id
salespeople.salesperson_id  ←→  orders.salesperson_id
```

---

## 2.2 INNER JOIN

An INNER JOIN returns only the rows where the join condition matches in **both** tables. Non-matching rows are dropped.

**Syntax:**

```sql
SELECT columns
FROM table_a
INNER JOIN table_b ON table_a.key = table_b.key;
```

**Example — show each order with the customer name:**

```sql
SELECT
    o.order_id,
    c.first_name,
    c.last_name,
    o.order_date,
    o.status
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_date;
```

**What is `o` and `c`?** They are table aliases — shorthand to avoid typing the full table name repeatedly. You define them after the table name (`FROM orders o`) and use them as a prefix (`o.order_id`).

**Expected:** 50 rows — every order matched to a customer name.

---

### Why aliases matter

Without aliases, column names from different tables can collide. Both `orders` and `customers` have a column called `customer_id`. If you write:

```sql
SELECT customer_id FROM orders INNER JOIN customers ON ...
```

PostgreSQL raises: `ERROR: column reference customer_id is ambiguous`

Always prefix columns with the table alias when joining:

```sql
SELECT o.order_id, c.customer_id, c.first_name
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id;
```

---

## 2.3 LEFT JOIN

A LEFT JOIN returns **all rows from the left table** and the matching rows from the right table. If there is no match, the right-table columns appear as NULL.

```sql
SELECT columns
FROM table_a                    -- left table: all rows included
LEFT JOIN table_b               -- right table: NULLs if no match
  ON table_a.key = table_b.key;
```

**Example — all customers, even those with no orders:**

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    o.order_id,
    o.order_date
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_id;
```

In our dataset every customer has at least one order, so all 50 order rows appear. But notice customer 1 (Alice) appears four times — once for each of her orders.

**Finding customers with no orders at all:**

```sql
SELECT c.customer_id, c.first_name, c.last_name
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;
```

Expected: 0 rows (all 20 customers have at least one order in this dataset). The technique is correct — try it on real data and it finds genuine gaps.

---

### INNER vs LEFT — when to use which

| Question | Use |
|----------|-----|
| What did customers buy? (only customers who bought something) | INNER JOIN |
| Which customers have NOT bought anything? | LEFT JOIN + WHERE right.id IS NULL |
| Show all products, even if never ordered | LEFT JOIN products to order_items |
| Revenue per salesperson (including $0 salespeople) | LEFT JOIN salespeople to orders |

In analytics, **LEFT JOIN is more common than INNER JOIN** because you usually want to see all entities even when some have no activity.

---

## 2.4 Joining More Than Two Tables

You can chain multiple JOINs in one query. Each JOIN adds one more table. PostgreSQL processes them left to right.

**Example — orders with customer name AND salesperson name:**

```sql
SELECT
    o.order_id,
    o.order_date,
    c.first_name  || ' ' || c.last_name  AS customer_name,
    s.first_name  || ' ' || s.last_name  AS salesperson_name,
    o.status
FROM orders o
INNER JOIN customers    c ON o.customer_id    = c.customer_id
INNER JOIN salespeople  s ON o.salesperson_id = s.salesperson_id
ORDER BY o.order_date;
```

> `||` is the PostgreSQL string concatenation operator. `'Alice' || ' ' || 'Johnson'` produces `'Alice Johnson'`.

**Expected:** 50 rows with full names for both customer and salesperson.

---

**Example — order line items with product name and customer name:**

```sql
SELECT
    o.order_id,
    c.first_name || ' ' || c.last_name AS customer,
    p.product_name,
    p.category,
    oi.quantity,
    oi.unit_price,
    oi.quantity * oi.unit_price AS line_total
FROM order_items oi
INNER JOIN orders    o ON oi.order_id   = o.order_id
INNER JOIN customers c ON o.customer_id = c.customer_id
INNER JOIN products  p ON oi.product_id = p.product_id
ORDER BY o.order_id, p.product_name;
```

**Expected:** 81 rows — one per line item.

> `oi.quantity * oi.unit_price AS line_total` is a **computed column** — you can do arithmetic in SELECT just like you would in a spreadsheet formula. The result gets the alias `line_total`.

---

## 2.5 Other JOIN Types

You will see these less often but you should know they exist.

### RIGHT JOIN

The mirror image of LEFT JOIN — all rows from the **right** table, matching rows from the left.

```sql
SELECT p.product_name, oi.order_id
FROM order_items oi
RIGHT JOIN products p ON oi.product_id = p.product_id;
```

This returns all products, even if never ordered. Most engineers rewrite RIGHT JOINs as LEFT JOINs by swapping the table order — it reads more naturally left-to-right.

### FULL OUTER JOIN

All rows from both tables. NULLs fill in wherever there is no match on either side.

```sql
SELECT c.customer_id, o.order_id
FROM customers c
FULL OUTER JOIN orders o ON c.customer_id = o.customer_id;
```

Useful when you want to find rows that exist in one table but not the other, in both directions at once.

### CROSS JOIN

Every row from table A combined with every row from table B. 20 customers × 15 products = 300 rows. Use deliberately and rarely — an accidental CROSS JOIN on large tables can crash a database.

---

## 2.6 Common Mistakes

### Forgetting the ON condition

```sql
-- Wrong: missing ON — produces a CROSS JOIN (every combination)
SELECT * FROM orders, customers;

-- Right:
SELECT * FROM orders o INNER JOIN customers c ON o.customer_id = c.customer_id;
```

### Joining on the wrong column

```sql
-- Wrong: customer_id joined to salesperson_id — produces nonsense rows
FROM orders o INNER JOIN salespeople s ON o.customer_id = s.salesperson_id

-- Right:
FROM orders o INNER JOIN salespeople s ON o.salesperson_id = s.salesperson_id
```

### Duplicate rows from a one-to-many JOIN

Joining `customers` to `orders` multiplies customer rows — each customer row repeats once per order. This is expected and correct. Be careful when counting: `COUNT(*)` counts rows, not unique customers.

```sql
-- Counts orders (correct for most questions)
SELECT COUNT(*) FROM customers c INNER JOIN orders o ON c.customer_id = o.customer_id;

-- Counts distinct customers who placed at least one order
SELECT COUNT(DISTINCT c.customer_id) FROM customers c INNER JOIN orders o ON c.customer_id = o.customer_id;
```

---

## 2.7 Key Takeaways

1. Tables are linked by **shared columns** — a primary key in one table and a foreign key in another.
2. **INNER JOIN** returns only matching rows from both tables.
3. **LEFT JOIN** returns all rows from the left table — NULLs appear where there is no match on the right.
4. You can chain as many JOINs as needed — each adds one more table.
5. Always use **table aliases** (`o`, `c`, `p`) and prefix all column names to avoid ambiguity.
6. A `||` concatenates strings in PostgreSQL.
7. Joining a one-to-many relationship **multiplies rows** — that is expected. Use `COUNT(DISTINCT ...)` when you need unique counts.
8. Most RIGHT JOINs can be rewritten as LEFT JOINs — prefer LEFT for readability.